# Chapter 10: Evaluation and Monitoring


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 10.1 Evaluation Framework


In [ ]:
from scratch_agent.eval.prompts import (
    ANSWER_RELEVANCY_PROMPT,
    CITATION_RELIABILITY_PROMPT,
    REQUIREMENT_COMPLIANCE_PROMPT,
    EVALUATION_SYSTEM_PROMPT,
)

print("Evaluation prompts loaded successfully.")
print(f"Answer Relevancy prompt length: {len(ANSWER_RELEVANCY_PROMPT)} chars")
print(f"Citation Reliability prompt length: {len(CITATION_RELIABILITY_PROMPT)} chars")
print(f"Requirement Compliance prompt length: {len(REQUIREMENT_COMPLIANCE_PROMPT)} chars")
print(f"\nAnswer Relevancy prompt:\n{ANSWER_RELEVANCY_PROMPT}")

## 10.2 Answer Relevancy


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.eval.prompts import ANSWER_RELEVANCY_PROMPT, EVALUATION_SYSTEM_PROMPT

# Use the evaluation prompt directly with LLM
llm = LlmClient(model="gpt-4o-mini")

question = "What is the capital of France?"
answer = "Paris is the capital of France."

# Fill in the prompt template
filled_prompt = ANSWER_RELEVANCY_PROMPT.format(question=question, answer=answer)

from scratch_agent.llm import LlmRequest
from scratch_agent.types import Message

request = LlmRequest(
    instructions=[EVALUATION_SYSTEM_PROMPT],
    contents=[Message(role="user", content=filled_prompt)],
)

response = await llm.generate(request)
print("Answer Relevancy Evaluation:")
print(response.content[0].content)

## 10.3 Citation Reliability


In [ ]:
from scratch_agent.eval.prompts import CITATION_RELIABILITY_PROMPT, EVALUATION_SYSTEM_PROMPT
from scratch_agent.llm import LlmClient, LlmRequest
from scratch_agent.types import Message

llm = LlmClient(model="gpt-4o-mini")

question = "How tall is the Eiffel Tower?"
answer = "The Eiffel Tower is 330 meters tall. It was built in 1889."
sources = "Wikipedia: https://en.wikipedia.org/wiki/Eiffel_Tower"

filled_prompt = CITATION_RELIABILITY_PROMPT.format(
    question=question, answer=answer, sources=sources
)

request = LlmRequest(
    instructions=[EVALUATION_SYSTEM_PROMPT],
    contents=[Message(role="user", content=filled_prompt)],
)

response = await llm.generate(request)
print("Citation Reliability Evaluation:")
print(response.content[0].content)

## 10.4 Requirement Compliance


In [ ]:
from scratch_agent.eval.prompts import REQUIREMENT_COMPLIANCE_PROMPT, EVALUATION_SYSTEM_PROMPT
from scratch_agent.llm import LlmClient, LlmRequest
from scratch_agent.types import Message

llm = LlmClient(model="gpt-4o-mini")

question = "Explain machine learning briefly."
requirements = "1. Under 100 words\n2. Include at least one example\n3. In English"
answer = "Machine learning is a subset of AI where systems learn from data. For example, spam filters use ML to classify emails as spam or not spam."

filled_prompt = REQUIREMENT_COMPLIANCE_PROMPT.format(
    question=question, requirements=requirements, answer=answer
)

request = LlmRequest(
    instructions=[EVALUATION_SYSTEM_PROMPT],
    contents=[Message(role="user", content=filled_prompt)],
)

response = await llm.generate(request)
print("Requirement Compliance Evaluation:")
print(response.content[0].content)

## 10.5 Overall Pass Rate (OPR)


In [ ]:
# Calculate Overall Pass Rate from multiple evaluation results
evaluation_results = [
    {"metric": "answer_relevancy", "score": 0.95, "passed": True},
    {"metric": "citation_reliability", "score": 0.80, "passed": True},
    {"metric": "requirement_compliance", "score": 0.60, "passed": False},
    {"metric": "answer_relevancy", "score": 1.00, "passed": True},
    {"metric": "citation_reliability", "score": 0.45, "passed": False},
]

total = len(evaluation_results)
passed = sum(1 for r in evaluation_results if r["passed"])
opr = passed / total

print(f"Total evaluations: {total}")
print(f"Passed: {passed}")
print(f"Failed: {total - passed}")
print(f"Overall Pass Rate (OPR): {opr:.1%}")

# Breakdown by metric
from collections import defaultdict

by_metric = defaultdict(list)
for r in evaluation_results:
    by_metric[r["metric"]].append(r["passed"])

print("\nBreakdown by metric:")
for metric, results in by_metric.items():
    metric_opr = sum(results) / len(results)
    print(f"  {metric}: {metric_opr:.1%} ({sum(results)}/{len(results)})")


## 10.6 GAIA Benchmark Final Evaluation


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import tool
from scratch_agent.eval.gaia import is_correct

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Create agent for evaluation
agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator],
    instructions="You are a general AI assistant. Answer questions accurately and concisely.",
    max_steps=5,
)

# Simple evaluation examples (without GAIA dataset)
test_cases = [
    {"question": "What is 7 * 8?", "answer": "56"},
    {"question": "What is 15 + 27?", "answer": "42"},
    {"question": "What is 100 / 4?", "answer": "25"},
]

correct_count = 0
for tc in test_cases:
    result = await agent.run(tc["question"])
    prediction = str(result.output) if result.output else ""
    correct = is_correct(prediction, tc["answer"])
    correct_count += correct
    print(f"Q: {tc['question']}")
    print(f"  Prediction: {prediction[:80]}")
    print(f"  Answer: {tc['answer']}")
    print(f"  Correct: {correct}")

print(f"\nAccuracy: {correct_count}/{len(test_cases)} ({correct_count/len(test_cases)*100:.0f}%)")